# PF2e Spell Tag Prediction Model

## Iteration 0: Baseline Dataset and Model

This notebook is the first iteration of the project. Its purpose is to prepare the Pathfinder 2e spell dataset and create a simple baseline model. The baseline gives us a reference point that can be used to compare future improvements in preprocessing, feature extraction, model architecture, and evaluation.

In [1]:
import pandas as pd
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import joblib

## Loading and preprocessing the data

The data is loaded from `spells.json`. The code supports both a JSON dictionary containing a spell list and a JSON file containing a list directly. For every spell, the description is stored as the input text, while its traits and school are combined into a list of labels. Spells without a description or labels are removed because they cannot provide enough information for supervised learning.

The resulting DataFrame contains the usable examples for the first model iteration.

In [2]:
# Load the raw spell data from the JSON file.
with open('spells.json', 'r', encoding='utf-8') as file:
    raw_data = json.load(file)

# Support both dictionary-based and list-based JSON structures.
if isinstance(raw_data, dict):
    keys = list(raw_data.keys())
    spell_list = raw_data[keys[0]] 
else:
    spell_list = raw_data


parsed_data = []

# Extract the description and labels from every spell.
for spell in spell_list:
    desc = spell.get('description', spell.get('text', ''))

    # Keep traits in a consistent list format.
    tags = spell.get('traits', [])
    if not isinstance(tags, list):
        tags = [tags] if pd.notna(tags) else []
        
    # Add the spell school as an additional label.
    school = spell.get('school', '')
    if school:
        tags.append(school)
        
    # Keep only examples containing both text and labels.
    if desc and tags:
        parsed_data.append({'text': desc, 'tags': tags})

# Create a structured DataFrame for analysis and model training.
df = pd.DataFrame(parsed_data)

# This is the number of usable training examples after preprocessing.
print(f"amount of spells: {len(df)}")

amount of spells: 1403


## Training data representation and KNN baseline

The spell descriptions are converted into numerical TF-IDF features. TF-IDF gives higher values to words that are important in a description and less importance to common English words. The labels are converted into a multi-label binary format because one spell can have several traits.

The four spell traditions (`arcane`, `occult`, `primal`, and `divine`) are excluded from the target labels in this baseline. The data is split into training and test sets, allowing the model to be evaluated on spells that it has not seen during training.

A K-Nearest Neighbours (KNN) classifier is used as a simple first model. Its results are not the final quality target; they provide a baseline for future iterations.

In [3]:

X = df['text'].str.replace('<[^<]+?>', '', regex=True) 
excluded_tags = {'arcane', 'occult', 'primal', 'divine'}
y = df['tags'].apply(
    lambda tags: [tag for tag in (tags or []) if tag.casefold() not in excluded_tags]
)

vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)
mlb = MultiLabelBinarizer()

X_encoded = vectorizer.fit_transform(X)
y_encoded = mlb.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42
)

print(f"Amount of features: {X_encoded.shape[1]}")
print(f"Amount of unique tags: {len(mlb.classes_)}")

Amount of features: 3000
Amount of unique tags: 103


In [4]:
# Train the baseline KNN model and evaluate it on unseen spells.
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

predictions = knn.predict(X_test)

f1_micro = f1_score(y_test, predictions, average='micro')
accuracy = accuracy_score(y_test, predictions)

print(f"Baseline Accuracy (amount of correct predictions): {accuracy:.2f}")
print(f"Baseline F1-Score (micro): {f1_micro:.2f}")

# Accuracy is strict: every label must be predicted correctly.
# Micro F1 evaluates all predicted labels together and is more informative
# for this multi-label problem. These values are the starting point for
# comparing future model iterations.

Baseline Accuracy (amount of correct predictions): 0.07
Baseline F1-Score (micro): 0.38


In [5]:
# Test the model on a new spell description that was not in the dataset.
custom_spell = [
    "A massive ball of superheated plasma explodes, dealing fire damage to anyone in the area."
]
custom_vec = vectorizer.transform(custom_spell)
custom_pred = knn.predict(custom_vec)

print("\ntest spell", custom_spell[0])
print("tags predicted:", mlb.inverse_transform(custom_pred))

# The prediction shows which learned tag is most similar to the description.
# This single example is qualitative evidence, not a complete model evaluation.

#joblib.dump(knn, 'knn_model.joblib')
#joblib.dump(vectorizer, 'vectorizer.joblib')
#joblib.dump(mlb, 'mlb.joblib')

#print("\n")


test spell A massive ball of superheated plasma explodes, dealing fire damage to anyone in the area.
tags predicted: [('evocation',)]
